In [ ]:
! cp ~/.kaggle/

cp: missing destination file operand after '/root/.kaggle/'
Try 'cp --help' for more information.


In [5]:
%pip -q install -U transformers accelerate bitsandbytes \
  "huggingface_hub>=0.34.0,<1.0" \
  sentence-transformers faiss-cpu tqdm rank-bm25 gradio

from google.colab import drive
drive.mount("/content/drive")

#DATA_DIR = "/content/drive/MyDrive/PIP_2025-2026_Groupe-1_Concours/Wahib B/cleaned_json_full"
# DATA_DIR = "/kaggle/input/cleaned-json-c2"

from huggingface_hub import notebook_login
notebook_login()

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 137.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 MB 41.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.8/23.8 MB 118.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.0/23.0 MB 122.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.7/55.7 kB 5.9 MB/s eta 0:00:00
Mounted at /content/drive


# Doc actuelle

In [16]:
import os

print("Contenu de /content/drive :\n")
try:
    for item in os.listdir('/content/drive/MyDrive/Telina/faiss_store'):
        print(f"- {item}")
except FileNotFoundError:
    print("Erreur : Le dossier /content/drive n'a pas été trouvé. Assurez-vous que Google Drive est monté.")
except Exception as e:
    print(f"Une erreur inattendue est survenue : {e}")

Contenu de /content/drive :

- index.faiss
- embeddings.npy
- texts.pkl


In [6]:
import os
from pathlib import Path

#DATA_DIR = "/content/drive/MyDrive/PIP_2025-2026_Groupe-1_Concours/Wahib B/cleaned_json_full"
DATA_DIR = "/content/drive/MyDrive/Telina/cleaned_json_full"

print("DATA_DIR =", DATA_DIR)
print("Existe ?", os.path.exists(DATA_DIR))

# Lister rapidement le contenu
print("\nContenu (20 premiers):")
print(os.listdir(DATA_DIR)[:20])

# Compter les fichiers JSON (récursif)
json_files = sorted([str(p) for p in Path(DATA_DIR).rglob("*.json")])
json_files_upper = sorted([str(p) for p in Path(DATA_DIR).rglob("*.JSON")])
print("\nNb .json :", len(json_files))
print("Nb .JSON :", len(json_files_upper))

# Montrer un exemple
example_list = json_files or json_files_upper
print("\nExemple:", example_list[0] if example_list else "Aucun JSON trouvé")

DATA_DIR = /content/drive/MyDrive/Telina/cleaned_json_full
Existe ? True

Contenu (20 premiers):
['page_001.json', 'page_117.json', 'page_076.json', 'page_081.json', 'page_072.json', 'page_006.json', 'page_096.json', 'page_052.json', 'page_100.json', 'page_056.json', 'page_015.json', 'page_071.json', 'page_026.json', 'page_109.json', 'page_114.json', 'page_047.json', 'page_098.json', 'page_011.json', 'page_016.json', 'page_087.json']

Nb .json : 126
Nb .JSON : 0

Exemple: /content/drive/MyDrive/Telina/cleaned_json_full/instituts_cnrs.json


In [7]:
import json

path0 = (json_files or json_files_upper)[0]
with open(path0, "r", encoding="utf-8") as f:
    sample = json.load(f)

print("Fichier:", path0)
print("Type:", type(sample))
if isinstance(sample, dict):
    print("Keys:", list(sample.keys())[:50])
else:
    print("Longueur liste:", len(sample))
    print("Keys du premier item:", list(sample[0].keys())[:50])

Fichier: /content/drive/MyDrive/Telina/cleaned_json_full/instituts_cnrs.json
Type: <class 'dict'>
Keys: ['source_file', 'date', 'institutes', 'pages']


In [8]:
import os, json, re
from pathlib import Path

def norm(x) -> str:
    """
    Normalise n'importe quel type vers une string 'propre'.
    - str -> normalisation whitespace
    - list/dict -> json stringifié
    - autres -> str(...)
    """
    if x is None:
        return ""
    if isinstance(x, str):
        s = x
    elif isinstance(x, (dict, list)):
        s = json.dumps(x, ensure_ascii=False)
    else:
        s = str(x)
    return re.sub(r"\s+", " ", s).strip()

def guess_kind(filename: str) -> str:
    fn = filename.lower()
    if fn.startswith("page_"):
        return "concours"
    if "deroul" in fn or "déroul" in fn or "process" in fn:
        return "general_process"
    if "avantage" in fn or "remuner" in fn or "rémun" in fn or "accompagner" in fn or "carriere" in fn or "carrière" in fn:
        return "general_career"
    if "institut" in fn or "cnrs" in fn:
        return "general_cnrs"
    return "general_other"

In [9]:
def extract_passages(obj: dict, filename: str):
    kind = guess_kind(filename)
    source = obj.get("url") or obj.get("source_url") or obj.get("source_file") or f"local://{filename}"

    passages = []

    # 1) Format "pages": liste de pages/sections
    if isinstance(obj.get("pages"), list):
        for i, p in enumerate(obj["pages"], start=1):
            # p peut être dict ou str
            if isinstance(p, dict):
                sec = p.get("title") or p.get("heading") or p.get("section") or f"Page {i}"
                txt = p.get("text") or p.get("content") or p.get("body") or ""
            else:
                sec = f"Page {i}"
                txt = str(p)
            txt = norm(txt)
            if txt:
                passages.append({
                    "text": txt,
                    "source": source,
                    "section": norm(sec),
                    "doc_id": filename,
                    "title": obj.get("title") or filename,
                    "kind": kind
                })
        return passages

    # 2) Format "institutes": liste (institut CNRS)
    if isinstance(obj.get("institutes"), list):
        for inst in obj["institutes"]:
            if not isinstance(inst, dict):
                continue
            name = inst.get("name") or inst.get("title") or inst.get("acronym") or "Institut"
            desc = inst.get("description") or inst.get("text") or inst.get("content") or ""
            desc = norm(desc)
            if desc:
                passages.append({
                    "text": desc,
                    "source": source,
                    "section": f"Institut: {norm(name)}",
                    "doc_id": filename,
                    "title": obj.get("title") or "Instituts CNRS",
                    "kind": kind
                })
        return passages

    # 3) Format concours / général : champs texte classiques
    title = obj.get("title") or obj.get("intitule") or obj.get("nom") or filename
    # Certains JSON ont des sections structurées
    if isinstance(obj.get("sections"), list):
        for sec in obj["sections"]:
            if not isinstance(sec, dict):
                continue
            sec_title = sec.get("title") or sec.get("heading") or sec.get("section") or "Section"
            sec_text = sec.get("text") or sec.get("content") or sec.get("body") or ""
            sec_text = norm(sec_text)
            if sec_text:
                passages.append({
                    "text": sec_text,
                    "source": source,
                    "section": norm(sec_title),
                    "doc_id": filename,
                    "title": title,
                    "kind": kind
                })
        return passages

    # 4) Fallback texte brut
    text = obj.get("text") or obj.get("content") or obj.get("body") or obj.get("texte") or ""
    text = norm(text)
    if text:
        passages.append({
            "text": text,
            "source": source,
            "section": "Document",
            "doc_id": filename,
            "title": title,
            "kind": kind
        })
        return passages

    # 5) Dernier recours: stringify propre (évite de perdre des infos)
    blob = norm(json.dumps(obj, ensure_ascii=False))
    if blob:
        passages.append({
            "text": blob,
            "source": source,
            "section": "Document (json)",
            "doc_id": filename,
            "title": title,
            "kind": kind
        })
    return passages

In [10]:
json_paths = sorted([str(p) for p in Path(DATA_DIR).rglob("*.json")])

passages = []
kinds_count = {}

for path in json_paths:
    fn = os.path.basename(path)
    with open(path, "r", encoding="utf-8") as f:
        obj = json.load(f)
    if isinstance(obj, dict):
        ps = extract_passages(obj, fn)
    elif isinstance(obj, list):
        ps = []
        for item in obj:
            if isinstance(item, dict):
                ps.extend(extract_passages(item, fn))
    else:
        ps = []
    passages.extend(ps)

for p in passages:
    kinds_count[p["kind"]] = kinds_count.get(p["kind"], 0) + 1

print("✅ Passages total:", len(passages))
print("Répartition kinds:", kinds_count)

# exemples
for ex in passages[:3]:
    print("\n---", ex["kind"], "|", ex["doc_id"], "|", ex["section"])
    print("source:", ex["source"])
    print(ex["text"][:250], "...")

✅ Passages total: 137
Répartition kinds: {'general_cnrs': 4, 'concours': 122, 'general_career': 11}

--- general_cnrs | instituts_cnrs.json | Page 1
source: instituts du CNRS.pdf
21/07/2025 16:32 Disciplines | CNRS 10 janvier 2024 Explorant tous les domaines de la science, la recherche au CNRS est multidisciplinaire par essence. En croisant les regards, les méthodes et les expertises, son approche interdisciplinaire permet qu ...

--- general_cnrs | instituts_cnrs.json | Page 2
source: instituts du CNRS.pdf
21/07/2025 16:32 Disciplines | CNRS Il a pour mission de développer et de coordonner les recherches concernant l'élaboration de nouveaux composés, la compréhension de la réactivité chimique, l'élucidation toujours plus fine et la prédiction des relat ...

--- general_cnrs | instituts_cnrs.json | Page 3
source: instituts du CNRS.pdf
21/07/2025 16:32 Disciplines | CNRS Il a pour mission de développer les recherches sur l'homme, aussi bien comme producteur de langages ou de savoirs que

In [11]:
import re

def chunk_text(text: str, chunk_size=900, overlap=120):
    sents = re.split(r"(?<=[\.\!\?])\s+", text)
    chunks, cur = [], ""
    for s in sents:
        s = s.strip()
        if not s:
            continue
        if len(cur) + len(s) + 1 <= chunk_size:
            cur = (cur + " " + s).strip()
        else:
            if cur:
                chunks.append(cur)
            # overlap = fin du chunk précédent
            if overlap > 0 and chunks:
                tail = chunks[-1][-overlap:]
                cur = (tail + " " + s).strip()
            else:
                cur = s
    if cur:
        chunks.append(cur)
    return chunks

chunked = []
for p in passages:
    for c in chunk_text(p["text"], chunk_size=900, overlap=120):
        chunked.append({**p, "text": c})

print("✅ Chunks:", len(chunked))

✅ Chunks: 661


In [11]:
from sentence_transformers import SentenceTransformer
import numpy as np, faiss
from tqdm import tqdm

EMBED_ID = "BAAI/bge-m3"
embedder = SentenceTransformer(EMBED_ID)

texts = [c["text"] for c in chunked]

def embed_all(texts, batch_size=64):
    vecs = []
    for i in tqdm(range(0, len(texts), batch_size)):
        v = embedder.encode(texts[i:i+batch_size], normalize_embeddings=True, show_progress_bar=False)
        vecs.append(v)
    return np.vstack(vecs).astype("float32")

emb = embed_all(texts, batch_size=64)

index = faiss.IndexFlatIP(emb.shape[1])  # cosine via vecteurs normalisés
index.add(emb)

print("✅ FAISS size:", index.ntotal, "dim:", emb.shape[1])

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

100%|██████████| 11/11 [00:19<00:00,  1.79s/it]

✅ FAISS size: 661 dim: 1024


## Relance

In [14]:
# relance
import os
import pickle
import faiss
import numpy as np

DATA_DIR = "/content/drive/MyDrive/Telina/cleaned_json_full"
SAVE_DIR = "faiss_store"

SAVE_PATH = os.path.join(DATA_DIR, SAVE_DIR)

# 🔴 IMPORTANT : créer le dossier
os.makedirs(SAVE_PATH, exist_ok=True)

# sauvegarde FAISS
faiss.write_index(index, os.path.join(SAVE_PATH, "index.faiss"))

# sauvegarde embeddings
np.save(os.path.join(SAVE_PATH, "embeddings.npy"), emb)

# sauvegarde textes
with open(os.path.join(SAVE_PATH, "texts.pkl"), "wb") as f:
    pickle.dump(texts, f)

print("✅ Sauvegarde terminée")


NameError: name 'index' is not defined

In [17]:
import os
import pickle
import faiss
import numpy as np
from sentence_transformers import SentenceTransformer

DATA_DIR = "/content/drive/MyDrive/Telina/cleaned_json_full"
SAVE_DIR = "faiss_store"
SAVE_PATH = os.path.join(DATA_DIR, SAVE_DIR)

EMBED_ID = "BAAI/bge-m3"

# 1️⃣ Charger le modèle d'embedding (obligatoire pour les requêtes)
embedder = SentenceTransformer(
    EMBED_ID,
    cache_folder="/content/drive/MyDrive/models",
    device="cpu"  # ou "cuda" si GPU
)

# 2️⃣ Charger l'index FAISS
index = faiss.read_index(os.path.join(SAVE_PATH, "index.faiss"))

# 3️⃣ Charger les textes
with open(os.path.join(SAVE_PATH, "texts.pkl"), "rb") as f:
    texts = pickle.load(f)

# (optionnel) Charger les embeddings
emb = np.load(os.path.join(SAVE_PATH, "embeddings.npy"))

print("✅ FAISS chargé :", index.ntotal)


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/123 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/54.0 [00:00<?, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/191 [00:00<?, ?B/s]

✅ FAISS chargé : 661


In [25]:
from sentence_transformers import CrossEncoder

RERANK_ID = "BAAI/bge-reranker-v2-m3"
reranker = CrossEncoder(RERANK_ID)
print("✅ Reranker chargé:", RERANK_ID)

config.json:   0%|          | 0.00/795 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.27G [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/17.1M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/964 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

✅ Reranker chargé: BAAI/bge-reranker-v2-m3


In [18]:
def retrieve(query: str, k=10, pre_k=60, kind_filter=None):
    qv = embedder.encode([query], normalize_embeddings=True).astype("float32")
    scores, ids = index.search(qv, pre_k)

    cand = []
    for s, idx in zip(scores[0], ids[0]):
        c = chunked[int(idx)]
        if kind_filter and c["kind"] not in kind_filter:
            continue
        cand.append(c)

    if not cand:
        return []

    # rerank
    pairs = [(query, c["text"]) for c in cand]
    rr = reranker.predict(pairs)

    ranked = sorted(zip(rr, cand), key=lambda x: x[0], reverse=True)[:k]
    out = []
    for rr_score, c in ranked:
        out.append({
            "score": float(rr_score),
            "text": c["text"],
            "source": c["source"],
            "section": c["section"],
            "doc_id": c["doc_id"],
            "kind": c["kind"],
        })
    return out

# petit test
res = retrieve("conditions d'accès au concours ingénieur CNRS", k=5, kind_filter={"general_career", "general_cnrs"})
for r in res:
    print(r["score"], r["doc_id"], r["section"])

NameError: name 'embedder' is not defined

In [15]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig

LLM_ID = "meta-llama/Meta-Llama-3.1-8B-Instruct"

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

llm_tok = AutoTokenizer.from_pretrained(LLM_ID, use_fast=True)
llm = AutoModelForCausalLM.from_pretrained(
    LLM_ID,
    quantization_config=bnb,
    device_map="auto",
    torch_dtype=torch.float16,
)

import torch

# à faire une fois
if llm_tok.pad_token_id is None:
    llm_tok.pad_token = llm_tok.eos_token

@torch.inference_mode()
def llama_chat(system: str, user: str, max_new_tokens=450, do_sample=True, temperature=0.1, top_p=0.9):
    msgs = [{"role":"system","content":system},{"role":"user","content":user}]
    enc = llm_tok.apply_chat_template(
        msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True
    )
    input_ids = enc["input_ids"].to(llm.device)
    attention_mask = enc["attention_mask"].to(llm.device)

    gen_kwargs = dict(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=max_new_tokens,
        eos_token_id=llm_tok.eos_token_id,
        pad_token_id=llm_tok.pad_token_id,
        do_sample=do_sample,
    )
    # éviter “flags ignorés” quand do_sample=False
    if do_sample:
        gen_kwargs.update(dict(temperature=temperature, top_p=top_p))

    out = llm.generate(**gen_kwargs)
    gen = out[0][input_ids.shape[-1]:]
    return llm_tok.decode(gen, skip_special_tokens=True).strip()

print("✅ Llama chargé")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(
`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

✅ Llama chargé


In [23]:
import re

SYSTEM = """Tu es un agent conversationnel RAG sur des documents CNRS (concours ingénieur).
Règles STRICTES anti-fausses-informations:
- Réponds en français.
- Utilise UNIQUEMENT le CONTEXTE. Ne complète jamais avec ta mémoire.
- Si tu n'es pas sûr à partir du CONTEXTE, dis: "Je ne peux pas répondre de manière fiable avec les documents disponibles."
- Cite tes sources sous forme [n] dans le texte (au moins à la fin de chaque point important).
- Ne fabrique jamais : dates, lieux, conditions, montants, intitulés.
- Ne mets pas de section "Sources" (elle sera ajoutée après).
"""

def format_context(passages, max_chars=420):
    lines=[]
    for i,p in enumerate(passages,1):
        excerpt = p["text"][:max_chars].strip() + ("..." if len(p["text"])>max_chars else "")
        lines.append(f"[{i}] ({p['source']} — {p['section']})\n{excerpt}")
    return "\n\n".join(lines)

def build_sources(passages, max_items=10):
    lines=["Sources:"]
    seen=set()
    for i,p in enumerate(passages,1):
        key=(p["source"], p["section"])
        if key in seen:
            continue
        seen.add(key)
        lines.append(f"- [{i}] {p['source']} | Section: {p['section']}")
        if len(seen) >= max_items:
            break
    return "\n".join(lines)

def detect_mode(q: str):
    ql = q.lower()
    if any(w in ql for w in ["je suis", "profil", "compétence", "competence", "qualif", "qualification", "stack", "candidat", "orienter", "quel concours"]):
        return "orient"
    if any(w in ql for w in ["carrière", "carriere", "grade", "rémun", "remun", "avantage", "prime", "branches", "métier", "metier"]):
        return "career"
    return "concours_info"

def answer_question(question: str, k=10, min_score=0.15):
    mode = detect_mode(question)

    if mode == "career":
        kind_filter = {"general_career","general_cnrs"}
    elif mode == "orient":
        kind_filter = {"concours"}
    else:
        kind_filter = None

    passages = retrieve(question, k=k, kind_filter=kind_filter)
    if not passages or passages[0]["score"] < min_score:
        return "Je ne peux pas répondre de manière fiable avec les documents disponibles.", []

    ctx = format_context(passages)

    user = f"""MODE: {mode}
QUESTION:
{question}

CONTEXTE:
{ctx}

Consigne de sortie:
- Si MODE=orient : propose 3 à 5 concours pertinents (titre/identifiant) + raison + citations.
- Sinon : réponds de façon structurée (puces) + citations.
"""
    ans = llama_chat(SYSTEM, user)
    final = ans.strip() + "\n" + build_sources(passages)
    return final, passages

In [18]:
# Interface chat basique (Notebook)

chat_history = []  # optionnel si tu veux stocker les échanges ici

def chat_loop():
    print("=== Chatbot RAG CNRS (Notebook) ===")
    print("Commandes: /quit pour quitter, /reset pour vider l'historique\n")

    while True:
        q = input("Vous: ").strip()
        if not q:
            continue

        if q.lower() in ["/quit", "quit", "exit"]:
            print("Fin du chat.")
            break

        if q.lower() in ["/reset", "reset"]:
            chat_history.clear()
            print("✅ Historique réinitialisé.\n")
            continue

        ans, _ = answer_question(q)
        chat_history.append(("Vous", q))
        chat_history.append(("Assistant", ans))

        print("\nAssistant:\n" + ans + "\n")

chat_loop()

=== Chatbot RAG CNRS (Notebook) ===
Commandes: /quit pour quitter, /reset pour vider l'historique

Vous: quit
Fin du chat.


In [11]:
import re

def detect_smalltalk(text: str):
    t = text.lower().strip()

    greet = {"bonjour","salut","hello","bonsoir","coucou"}
    bye = {"au revoir","aurevoir","bye","à bientôt","a bientot","bonne journée","bonne soiree","bonne soirée"}
    thanks = {"merci","merci beaucoup","thx","thanks","je te remercie"}

    if t in greet or re.match(r"^(bonjour|salut|hello|bonsoir)\b", t):
        return "greet"
    if t in thanks or re.match(r"^merci\b", t):
        return "thanks"
    if t in bye or re.match(r"^(au revoir|bye)\b", t):
        return "bye"
    if t in {"aide","help","?"}:
        return "help"
    return None

def smalltalk_response(intent: str):
    if intent == "greet":
        return ("Bonjour 👋 Je peux t’aider à :\n"
                "- trouver des concours ingénieur CNRS selon tes compétences\n"
                "- expliquer un concours précis\n"
                "- donner des infos sur les carrières (grades, avantages, rémunération si présent)\n"
                "- expliquer le déroulement des concours (si présent dans les docs)\n\n"
                "Dis-moi ton profil (compétences, domaine) ou le concours qui t’intéresse.")
    if intent == "thanks":
        return "Avec plaisir ! Si tu veux, décris ton profil (compétences/expérience) et je te propose des concours pertinents."
    if intent == "bye":
        return "Au revoir ! N’hésite pas à revenir si tu as d’autres questions sur les concours CNRS."
    if intent == "help":
        return ("Tu peux me demander par exemple :\n"
                "- « Je suis ingénieur data, quels concours me correspondent ? »\n"
                "- « Donne-moi les missions/compétences du concours page_072 »\n"
                "- « Quels sont les avantages à travailler au CNRS ? »\n"
                "- « Quelles sont les phases du concours ? »")
    return "Je suis là 🙂"

In [13]:
def rewrite_query(user_question: str, history_pairs, max_len=220):
    ctx = "\n".join([f"User: {u}\nAssistant: {a}" for u,a in history_pairs[-2:]])
    prompt = f"""Tu es un assistant qui reformule des questions pour un moteur de recherche documentaire.
Reformule la QUESTION en une requête autonome, courte et précise, en français.
N'ajoute pas d'information. Ne réponds pas à la question.

HISTORIQUE (optionnel):
{ctx}

QUESTION:
{user_question}

Requête reformulée:"""

    msgs = [
        {"role":"system","content":"Tu reformules des requêtes."},
        {"role":"user","content":prompt},
    ]
    enc = llm_tok.apply_chat_template(
        msgs, add_generation_prompt=True, return_tensors="pt", return_dict=True
    )
    input_ids = enc["input_ids"].to(llm.device)
    attention_mask = enc["attention_mask"].to(llm.device)

    out = llm.generate(
        input_ids=input_ids,
        attention_mask=attention_mask,
        max_new_tokens=80,
        do_sample=False,   # ✅ greedy decoding
        eos_token_id=llm_tok.eos_token_id,
        pad_token_id=llm_tok.pad_token_id,
    )

    gen = out[0][input_ids.shape[-1]:]
    q = llm_tok.decode(gen, skip_special_tokens=True).strip()
    return q.replace("\n"," ")[:max_len]

In [20]:
from collections import defaultdict

def group_by_doc(passages):
    grouped = defaultdict(list)
    for p in passages:
        grouped[p["doc_id"]].append(p)
    return grouped

In [9]:
chat_pairs = []  # [(user, assistant), ...]

SYSTEM_STRICT = """Tu es un agent conversationnel RAG sur des documents CNRS (concours ingénieur).
Règles STRICTES:
- Réponds en français, avec des phrases correctes et un ton naturel.
- Utilise UNIQUEMENT le CONTEXTE fourni. Ne complète jamais avec ta mémoire.
- Si tu ne peux pas répondre de façon fiable à partir du contexte, dis-le clairement.
- Ne JAMAIS inventer : dates, lieux, conditions, montants, intitulés.
- Mets des citations [n] dans le texte pour les infos importantes.
- Ne mets pas "Sources:" (ajouté automatiquement).
"""

def format_context(passages, max_chars=420):
    lines=[]
    for i,p in enumerate(passages,1):
        excerpt = p["text"][:max_chars].strip() + ("..." if len(p["text"])>max_chars else "")
        lines.append(f"[{i}] ({p['source']} — {p['section']} — {p['doc_id']})\n{excerpt}")
    return "\n\n".join(lines)

def build_sources_used(answer_text, passages, max_items=12):
    used = sorted(set(int(n) for n in re.findall(r"\[(\d+)\]", answer_text)))
    lines=["Sources:"]
    seen=set()
    count=0
    for n in used:
        if 1 <= n <= len(passages):
            p = passages[n-1]
            key=(p["source"], p["section"])
            if key in seen:
                continue
            seen.add(key)
            lines.append(f"- [{n}] {p['source']} | Section: {p['section']}")
            count += 1
            if count >= max_items:
                break
    # si aucune citation, on met quand même les 3 meilleures sources
    if len(lines) == 1 and passages:
        for i,p in enumerate(passages[:3],1):
            lines.append(f"- [{i}] {p['source']} | Section: {p['section']}")
    return "\n".join(lines)

def detect_mode(q: str):
    ql = q.lower()
    if any(w in ql for w in ["je suis", "profil", "compétence", "competence", "qualification", "orient", "quel concours", "correspond"]):
        return "orient"
    if any(w in ql for w in ["carrière","carriere","grade","rémun","remun","avantage","prime","branches","métier","metier"]):
        return "career"
    if any(w in ql for w in ["phase","dérou","derou","audition","jury","calendrier","date","lieu","épreuve","conditions"]):
        return "process_or_rules"
    return "concours_info"

def chatbot_respond(user_text: str, k=12, min_score=0.15):
    global chat_pairs

    # 1) smalltalk
    intent = detect_smalltalk(user_text)
    if intent:
        ans = smalltalk_response(intent)
        chat_pairs.append((user_text, ans))
        return ans

    # 2) rewrite query for better retrieval
    rq = rewrite_query(user_text, chat_pairs)

    # 3) mode -> kind_filter (avec ce qu'on a dans tes JSON)
    mode = detect_mode(user_text)
    if mode == "career":
        kind_filter = {"general_career", "general_cnrs"}
    elif mode == "orient":
        kind_filter = {"concours"}
    else:
        kind_filter = None

    # 4) retrieve
    passages = retrieve(rq, k=k, kind_filter=kind_filter)

    if not passages or passages[0]["score"] < min_score:
        ans = ("Je ne peux pas répondre de manière fiable avec les documents disponibles.\n"
               "Tu peux reformuler, ou me donner le nom/ID du concours (ex: page_072) si tu en as un.")
        chat_pairs.append((user_text, ans))
        return ans

    ctx = format_context(passages)

    # 5) génération (réponse structurée)
    if mode == "orient":
        user_prompt = f"""Tu dois aider à orienter vers les concours pertinents.
À partir du CONTEXTE, propose 3 à 5 concours (doc_id) maximum.
Pour chacun: titre si visible, pourquoi (compétences/mission) + citations [n].
Si l'info est absente, dis-le.

QUESTION:
{user_text}

CONTEXTE:
{ctx}
"""
    else:
        user_prompt = f"""Réponds de façon claire et structurée (phrases correctes, puces si utile).
Si la réponse est partielle, ajoute une section "Limites" (1-2 lignes).
QUESTION:
{user_text}

CONTEXTE:
{ctx}
"""

    ans = llama_chat(SYSTEM_STRICT, user_prompt, max_new_tokens=520, temperature=0.1)
    final = ans.strip() + "\n" + build_sources_used(ans, passages)

    chat_pairs.append((user_text, final))
    return final

In [8]:
def chat_loop():
    print("=== Chatbot RAG CNRS (Notebook) ===")
    print("Commandes: /quit, /reset\n")
    while True:
        q = input("Vous: ").strip()
        if not q:
            continue
        if q.lower() in ["/quit", "quit", "exit"]:
            print("Fin du chat.")
            break
        if q.lower() in ["/reset", "reset"]:
            chat_pairs.clear()
            print("✅ Historique réinitialisé.\n")
            continue

        ans = chatbot_respond(q)
        print("\nAssistant:\n" + ans + "\n")

chat_loop()

=== Chatbot RAG CNRS (Notebook) ===
Commandes: /quit, /reset

Vous: quit
Fin du chat.


# 2 - Metrique

In [18]:
import torch
from transformers import pipeline

judge_pipe = pipeline(
    "text2text-generation",
    #model="google/flan-t5-base",
    model = "google/flan-t5-large",
    device=0 if torch.cuda.is_available() else -1
)



config.json:   0%|          | 0.00/662 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.13G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

Device set to use cuda:0


In [25]:
import os

print("Contenu de /content/drive :\n")
try:
    for item in os.listdir('/content/drive/MyDrive/Telina'):
        print(f"- {item}")
except FileNotFoundError:
    print("Erreur : Le dossier /content/drive n'a pas été trouvé. Assurez-vous que Google Drive est monté.")
except Exception as e:
    print(f"Une erreur inattendue est survenue : {e}")

Contenu de /content/drive :

- jeu_test.csv
- cleaned_json_full
- met_taha.ipynb


In [3]:
import pandas as pd

data = pd.read_csv("/content/drive/MyDrive/Telina/jeu_test.csv")
data.head()


,ID,Question,expected_answer,category,must_refuse,must_cite_source,expected_sources
0,1,Combien de postes sont disponibles pour le con...,Le concours comprend trois postes distincts :\...,concours_info,0.0,1.0,page_001.html
1,2,Est ce que le concours Experte ou expert en in...,"Pour le concours n°64, le grade mentionné est ...",orientation,0.0,1.0,Guide candidat 2025.pdf
2,3,Quelles sont les responsabilités de l'ingénieu...,L'agent développera des outils numériques avan...,concours_info,0.0,1.0,page_064.html
3,4,Quel salaire je peux prétendre en fin de carri...,Le concours numéro 64 fait référence à un pos...,career_general,0.0,1.0,Guide candidat 2025.pdf
4,5,Est ce que le lieu du concours Experte ou expe...,La fiche du concours indique que le lieu de tr...,process_general,1.0,0.0,NaN


In [7]:
# Pré-requis commun
# Fonction pour enlever “Sources:”
def strip_sources_block(text: str) -> str:
    if not text:
        return ""
    if "Sources:" in text:
        text = text.split("Sources:")[0]
    return text.strip()
# Fonction pour convertir passages → contexts pour les juges
def passages_to_contexts(passages):
    # transforme en format standard : [{"text": "..."}]
    return [{"text": p.get("text", "")} for p in passages]
# answer() standard pour l’éval
def answer_eval(question: str, k=12, min_score=0.15):
    # on appelle ta fonction existante
    ans = chatbot_respond(question, k=k, min_score=min_score)

    #  chatbot_respond ne renvoie pas les passages, donc on les récupère à part
    # ici on choisit retrieve(question) direct (sans rewrite)
    passages = retrieve(question, k=k)

    return ans, passages


# 1) Faithfulness (LLM-as-Judge)

In [28]:
import numpy as np
import pandas as pd

def evaluate_faithfulness(answer, contexts, judge_pipe, max_context_chars=2000):
    context_text = " ".join((c.get("text", "") or "") for c in contexts)[:max_context_chars]

    prompt = f"""
Cette réponse est-elle fidèle au contexte ?
Réponds uniquement par OUI ou NON.

Contexte :
{context_text}

Réponse :
{answer}

La réponse est-elle soutenue par le contexte ?
"""

    out = judge_pipe(prompt, max_new_tokens=5, do_sample=False)[0]["generated_text"].lower().strip()
    return 1 if out.startswith("oui") else 0


faithfulness_scores = []
all_results = []

for idx, row in data.iterrows():
    q = row["Question"]

    ans_text, passages = answer_eval(q)
    ans_only = strip_sources_block(ans_text)

    contexts = passages_to_contexts(passages)

    score = evaluate_faithfulness(
        answer=ans_only,
        contexts=contexts,
        judge_pipe=judge_pipe
    )

    faithfulness_scores.append(score)

    all_results.append({
        "ID": row.get("ID", idx),
        "Question": q,
        "Faithfulness": score
    })

results_df = pd.DataFrame(all_results)
results_df.to_csv("faithfulness_results_llama.csv", index=False)

print("✅ Faithfulness moyenne :", np.mean(faithfulness_scores))
results_df.head()


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
Token indices sequence length is longer than the specified maximum sequence length for this model (938 > 512). Running this sequence through the model will result in indexing errors
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


✅ Faithfulness moyenne : 0.0


,ID,Question,Faithfulness
0,1,Combien de postes sont disponibles pour le con...,0
1,2,Est ce que le concours Experte ou expert en in...,0
2,3,Quelles sont les responsabilités de l'ingénieu...,0
3,4,Quel salaire je peux prétendre en fin de carri...,0
4,5,Est ce que le lieu du concours Experte ou expe...,0


from matplotlib import pyplot as plt
_df_0['index'].plot(kind='hist', bins=20, title='index')
plt.gca().spines[['top', 'right',]].set_visible(False)

from matplotlib import pyplot as plt
_df_1['ID'].plot(kind='hist', bins=20, title='ID')
plt.gca().spines[['top', 'right',]].set_visible(False)

...

from matplotlib import pyplot as plt
_df_8['ID'].plot(kind='line', figsize=(8, 4), title='ID')
plt.gca().spines[['top', 'right']].set_visible(False)

<string>:5: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

) missing from font(s) DejaVu Sans.
  plt.savefig(


from matplotlib import pyplot as plt
import seaborn as sns
figsize = (12, 1.2 * len(_df_9['Question'].unique()))
plt.figure(figsize=figsize)
sns.violinplot(_df_9, x='index', y='Question', inner='stick', palette='Dark2')
sns.despine(top=True, right=True, bottom=True, left=True)

<string>:5: FutureWarning: 

Passing `palette` without assigning `hue` is deprecated and will be removed in v0.14.0. Assign the `y` variable to `hue` and set `legend=False` for the same effect.

) missing from font(s) DejaVu Sans.
  plt.savefig(


from matplotlib import pyplot as plt
import seaborn as sns
figsize = (12, 1.2 * len(_df_10['Question'].unique()))
plt.figure(figsize=figsize)
sns.violinplot(_df_10, x='ID', y='Question', inner='stick', palette='Dark2')
sns.despine(top=True, right=True, bottom=True, left=True)

In [30]:
results_df = pd.DataFrame(all_results)
results_df

,ID,Question,Faithfulness
0,1,Combien de postes sont disponibles pour le con...,0
1,2,Est ce que le concours Experte ou expert en in...,0
2,3,Quelles sont les responsabilités de l'ingénieu...,0
3,4,Quel salaire je peux prétendre en fin de carri...,0
4,5,Est ce que le lieu du concours Experte ou expe...,0
5,6,où et quand se déroule le concours Experte ou ...,0
6,7,Quelles sont les missions du poste ingénieur b...,0
7,8,Quels sont les pré-requis pour passer le conco...,0
8,9,Quel est le salaire net mensuel exact pour le ...,0
9,10,Quels sont les diplômes requis pour postuler a...,0


# 2) Answer Relevancy (LLM-as-Judge 1→5)

In [31]:
import re
import numpy as np
import pandas as pd

def evaluate_answer_relevancy(question, answer, judge_pipe, max_new_tokens=10):
    prompt = f"""
Donne une note de 1 à 5 uniquement (un seul chiffre).

Question :
{question}

Réponse :
{answer}

Pertinence (1 = hors sujet, 5 = répond parfaitement) :
"""

    out = judge_pipe(
        prompt,
        max_new_tokens=max_new_tokens,
        do_sample=False
    )[0]["generated_text"].strip()

    match = re.search(r"\b([1-5])\b", out)
    if not match:
        return 0.0

    return int(match.group(1)) / 5.0


relevancy_scores = []
all_results = []

for idx, row in data.iterrows():
    q = row["Question"]

    ans_text, _ = answer_eval(q)
    ans_only = strip_sources_block(ans_text)

    score = evaluate_answer_relevancy(
        question=q,
        answer=ans_only,
        judge_pipe=judge_pipe
    )

    relevancy_scores.append(score)

    all_results.append({
        "ID": row.get("ID", idx),
        "Question": q,
        "Answer_Relevancy_Judge": score
    })

results_df = pd.DataFrame(all_results)
results_df.to_csv("answer_relevancy_judge_llama.csv", index=False)

print("✅ Answer relevancy moyenne :", np.mean(relevancy_scores))
results_df.head()


✅ Answer relevancy moyenne : 0.1958333333333333


,ID,Question,Answer_Relevancy_Judge
0,1,Combien de postes sont disponibles pour le con...,0.2
1,2,Est ce que le concours Experte ou expert en in...,0.2
2,3,Quelles sont les responsabilités de l'ingénieu...,0.2
3,4,Quel salaire je peux prétendre en fin de carri...,0.2
4,5,Est ce que le lieu du concours Experte ou expe...,0.2


In [33]:
output_path = "/content/drive/MyDrive/Telina/answer_relevancy_judge_llama.csv"
results_df.to_csv(output_path, index=False)

print("✅ Fichier enregistré dans :", output_path)

✅ Fichier enregistré dans : /content/drive/MyDrive/Telina/answer_relevancy_judge_llama.csv


# 3) Context Precision (LLM-as-Judge)

In [34]:
import numpy as np
import pandas as pd

def evaluate_context_precision(question, contexts, judge_pipe, max_chunk_chars=800):
    if not contexts:
        return 0.0

    relevant = 0

    for c in contexts:
        chunk_text = (c.get("text", "") or "")[:max_chunk_chars]

        prompt = f"""
Réponds uniquement par OUI ou NON.

Question :
{question}

Contexte :
{chunk_text}

Ce contexte est-il pertinent pour répondre à la question ?
"""

        out = judge_pipe(
            prompt,
            max_new_tokens=3,
            do_sample=False
        )[0]["generated_text"].lower().strip()

        if out.startswith("oui"):
            relevant += 1

    return relevant / len(contexts)


context_precision_scores = []
all_results = []

for idx, row in data.iterrows():
    q = row["Question"]

    passages = retrieve(q, k=12)
    contexts = passages_to_contexts(passages)

    score = evaluate_context_precision(
        question=q,
        contexts=contexts,
        judge_pipe=judge_pipe
    )

    context_precision_scores.append(score)

    all_results.append({
        "ID": row.get("ID", idx),
        "Question": q,
        "Context_Precision": score
    })

results_df3 = pd.DataFrame(all_results)
results_df3.to_csv("context_precision_llama.csv", index=False)

print("✅ Context precision moyenne :", np.mean(context_precision_scores))
results_df3.head()

output_path = "/content/drive/MyDrive/Telina/evaluate_context_precision_llama.csv"
results_df.to_csv(output_path, index=False)

print("✅ Fichier enregistré dans :", output_path)


✅ Context precision moyenne : 0.003472222222222222
✅ Fichier enregistré dans : /content/drive/MyDrive/Telina/evaluate_context_precision_llama.csv


# 4) Context Recall (LLM-as-Judge + expected_answer)

In [30]:
import numpy as np
import pandas as pd

def evaluate_context_recall(ground_truth_answer, contexts, judge_pipe, max_context_chars=1500):
    if not contexts:
        return 0

    context_text = " ".join(
        (c.get("text", "") or "")[:400] for c in contexts
    )[:max_context_chars]

    prompt = f"""
Réponds uniquement par OUI ou NON.

Règle stricte :
- OUI = toutes les informations essentielles sont présentes dans le contexte
- NON = s'il manque au moins une information essentielle

Contexte :
{context_text}

Réponse attendue :
{ground_truth_answer}

Toutes les informations nécessaires pour produire EXACTEMENT cette réponse
sont-elles présentes dans le contexte ?
"""

    out = judge_pipe(
        prompt,
        max_new_tokens=3,
        do_sample=False
    )[0]["generated_text"].lower().strip()

    return 1 if out.startswith("oui") else 0


context_recall_scores = []
all_results = []

for idx, row in data.iterrows():
    q = row["Question"]
    gt = row["expected_answer"]

    passages = retrieve(q, k=12)
    contexts = passages_to_contexts(passages)

    score = evaluate_context_recall(
        ground_truth_answer=gt,
        contexts=contexts,
        judge_pipe=judge_pipe
    )

    context_recall_scores.append(score)

    all_results.append({
        "ID": row.get("ID", idx),
        "Question": q,
        "Context_Recall": score
    })

results_df4 = pd.DataFrame(all_results)
results_df4.to_csv("context_recall_llama.csv", index=False)

print("✅ Context recall moyen :", np.mean(context_recall_scores))
results_df4.head()

output_path = "/content/drive/MyDrive/Telina/evaluate_context_recall_llama.csv"
results_df4.to_csv(output_path, index=False)

print("✅ Fichier enregistré dans :", output_path)


Token indices sequence length is longer than the specified maximum sequence length for this model (857 > 512). Running this sequence through the model will result in indexing errors
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


KeyboardInterrupt: 

# 5) Refusal Accuracy (basé sur must_refuse)

In [16]:
import numpy as np
import pandas as pd
import re

REFUSAL_PATTERNS = [
    r"je ne peux pas répondre",
    r"je ne peux pas répondre de manière fiable",
    r"je ne dispose pas",
    r"je n'ai pas (?:cette|ces) information",
    r"aucune (?:source|information)",
]

def is_refusal(text: str) -> bool:
    if not text:
        return True
    t = text.lower().strip()
    return any(re.search(p, t) for p in REFUSAL_PATTERNS)


all_results = []
correct_list = []

for idx, row in data.iterrows():
    q = row["Question"]
    must_refuse = int(row["must_refuse"]) if pd.notna(row["must_refuse"]) else 0

    ans_text, _ = answer_eval(q)
    ans_only = strip_sources_block(ans_text)

    gt_refusal = (must_refuse == 1)
    ans_refusal = is_refusal(ans_only)

    correct = int(gt_refusal == ans_refusal)
    correct_list.append(correct)

    all_results.append({
        "ID": row.get("ID", idx),
        "Question": q,
        "must_refuse": must_refuse,
        "ans_refusal": int(ans_refusal),
        "Refusal_Correct": correct
    })

results_df5 = pd.DataFrame(all_results)
results_df5.to_csv("refusal_accuracy_llama.csv", index=False)

print("✅ Refusal Accuracy :", np.mean(correct_list))
results_df5.head()

output_path = "/content/drive/MyDrive/Telina/is_refusal.csv"
results_df5.to_csv(output_path, index=False)

print("✅ Fichier enregistré dans :", output_path)


The following generation flags are not valid and may be ignored: ['temperature', 'top_p']. Set `TRANSFORMERS_VERBOSITY=info` for more details.


NameError: name 'retrieve' is not defined

# 6) Retrieval Precision@k / Recall@k / F1@k

In [ ]:
import numpy as np
import pandas as pd
import re
import os

def precision_at_k(retrieved, relevant, k):
    top_k = set(retrieved[:k])
    relevant_retrieved = top_k & relevant
    return len(relevant_retrieved) / k if k > 0 else 0

def recall_at_k(retrieved, relevant, k):
    top_k = set(retrieved[:k])
    relevant_retrieved = top_k & relevant
    return len(relevant_retrieved) / len(relevant) if relevant else 0

def f1_at_k(p, r):
    if (p + r) == 0:
        return 0.0
    return 2 * p * r / (p + r)

def parse_expected_sources(x):
    if pd.isna(x):
        return set()
    parts = re.split(r"[;,]", str(x))
    return set(p.strip() for p in parts if p.strip())

def normalize_source_name(filename: str):
    if not filename:
        return ""
    base = os.path.basename(str(filename)).strip()
    return os.path.splitext(base)[0]


K = 10  # ou K=TOP_K

precision_scores = []
recall_scores = []
f1_scores = []
all_results = []

for idx, row in data.iterrows():
    q = row["Question"]

    # expected sources
    expected = parse_expected_sources(row["expected_sources"])
    expected_norm = set(normalize_source_name(s) for s in expected)

    # retrieved docs (doc_id)
    passages = retrieve(q, k=K)
    retrieved_doc_ids = [normalize_source_name(p.get("doc_id", "")) for p in passages]

    # unique doc ids (doc-level)
    retrieved_unique = []
    seen = set()
    for d in retrieved_doc_ids:
        if d and d not in seen:
            retrieved_unique.append(d)
            seen.add(d)

    p = precision_at_k(retrieved_unique, expected_norm, k=K)
    r = recall_at_k(retrieved_unique, expected_norm, k=K)
    f1 = f1_at_k(p, r)

    precision_scores.append(p)
    recall_scores.append(r)
    f1_scores.append(f1)

    all_results.append({
        "ID": row.get("ID", idx),
        "Question": q,
        f"Precision@{K}": p,
        f"Recall@{K}": r,
        f"F1@{K}": f1,
        "expected_sources": str(row["expected_sources"]),
        "retrieved_doc_ids": ", ".join(retrieved_unique)
    })

results_df = pd.DataFrame(all_results)
results_df.to_csv(f"retrieval_metrics_at_{K}_llama.csv", index=False)

print(f"✅ Precision@{K} moyenne :", np.mean(precision_scores))
print(f"✅ Recall@{K} moyen :", np.mean(recall_scores))
print(f"✅ F1@{K} moyen :", np.mean(f1_scores))

results_df.head()


# 7) Similarité réponses (Generated vs expected_answer) (Embeddings)

In [ ]:
import numpy as np
import pandas as pd
from sentence_transformers import util

def answer_similarity(generated_answer, expected_answer, embedder):
    gen_emb = embedder.encode(generated_answer, normalize_embeddings=True)
    exp_emb = embedder.encode(expected_answer, normalize_embeddings=True)
    return float(util.cos_sim(gen_emb, exp_emb).item())


sim_scores = []
all_results = []

for idx, row in data.iterrows():
    q = row["Question"]
    gt = row["expected_answer"]

    ans_text, _ = answer_eval(q)
    ans_only = strip_sources_block(ans_text)

    score = answer_similarity(
        generated_answer=ans_only,
        expected_answer=gt,
        embedder=embedder
    )

    sim_scores.append(score)

    all_results.append({
        "ID": row.get("ID", idx),
        "Question": q,
        "Answer_Similarity_GT": score
    })

results_df = pd.DataFrame(all_results)
results_df.to_csv("answer_similarity_gt_llama.csv", index=False)

print("✅ Similarité moyenne ANS vs GT :", np.mean(sim_scores))
results_df.head()


# 8) Answer Relevancy Embedding (Question vs Answer)

In [ ]:
import numpy as np
import pandas as pd
from sentence_transformers import util

def answer_relevancy_embedding(question, answer, embedder):
    q_emb = embedder.encode(question, normalize_embeddings=True)
    a_emb = embedder.encode(answer, normalize_embeddings=True)
    return float(util.cos_sim(q_emb, a_emb).item())


scores = []
all_results = []

for idx, row in data.iterrows():
    q = row["Question"]

    ans_text, _ = answer_eval(q)
    ans_only = strip_sources_block(ans_text)

    score = answer_relevancy_embedding(
        question=q,
        answer=ans_only,
        embedder=embedder
    )

    scores.append(score)

    all_results.append({
        "ID": row.get("ID", idx),
        "Question": q,
        "Answer_Relevancy_Embedding": score
    })

results_df = pd.DataFrame(all_results)
results_df.to_csv("answer_relevancy_embedding_llama.csv", index=False)

print("✅ Relevancy (embedding) moyenne :", np.mean(scores))
results_df.head()


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
import json
import pandas as pd

jsonl_path = "/content/drive/MyDrive/PIP_2025-2026_Groupe-1_Concours/Hosni Youssef/dataset_cnrs_FINAL_HYBRIDE.jsonl"
csv_path = "/content/drive/MyDrive/PIP_2025-2026_Groupe-1_Concours/Hosni Youssef/dataset_cnrs_FINAL_HYBRIDE.csv"

rows = []
with open(jsonl_path, "r", encoding="utf-8") as f:
    for line in f:
        line = line.strip()
        if line:
            rows.append(json.loads(line))

df = pd.DataFrame(rows)

# rendre lisibles les listes/dicts si jamais il y en a
for col in df.columns:
    df[col] = df[col].apply(
        lambda x: ", ".join(x) if isinstance(x, list)
        else json.dumps(x, ensure_ascii=False) if isinstance(x, dict)
        else x
    )

df.to_csv(csv_path, index=False, encoding="utf-8-sig")

print("CSV créé :", csv_path)
df.head()

CSV créé : /content/drive/MyDrive/PIP_2025-2026_Groupe-1_Concours/Hosni Youssef/dataset_cnrs_FINAL_HYBRIDE.csv


,instruction,input,output
0,Est-ce que ce concours correspond bien à mes c...,DOCUMENT SOURCE : page_001.json\n\nCONTENU :\n...,"Selon le document page_001.html, le poste d'In..."
1,Quand se déroulera le concours et où aura-t-il...,DOCUMENT SOURCE : page_001.json\n\nCONTENU :\n...,"Selon le document page_001.html, le concours n..."
2,Quelle est la mission principale de l'Ingénieu...,DOCUMENT SOURCE : page_001.json\n\nCONTENU :\n...,"Selon le document page_001.html, la mission pr..."
3,Est-ce que ce concours correspond bien à mon p...,DOCUMENT SOURCE : page_009.json\n\nCONTENU :\n...,"Selon le document page_009.html, ce concours e..."
4,Quand et où se déroule le concours n°9 pour le...,DOCUMENT SOURCE : page_009.json\n\nCONTENU :\n...,"Selon le document page_009.html, le concours n..."


In [ ]:
DOCX_PATH = "/content/drive/MyDrive/PIP_2025-2026_Groupe-1_Concours/Mohamed-Taha Belhaj - Analyse/questions .docx"

In [ ]:
import os
from docx import Document
import pandas as pd

DOCX_PATH = "/content/drive/MyDrive/PIP_2025-2026_Groupe-1_Concours/Mohamed-Taha Belhaj - Analyse/questions .docx"
OUT_CSV_PATH = "/content/drive/MyDrive/PIP_2025-2026_Groupe-1_Concours/Mohamed-Taha Belhaj - Analyse/questions_structured_template.csv"

print("DOCX existe ?", os.path.exists(DOCX_PATH))
if not os.path.exists(DOCX_PATH):
    raise FileNotFoundError(f"Fichier introuvable: {DOCX_PATH}")

doc = Document(DOCX_PATH)

# On récupère les paragraphes non vides
paras = [p.text.strip() for p in doc.paragraphs if p.text and p.text.strip()]
print("Nb paragraphes non vides:", len(paras))

# Hypothèse: alternance Question puis Réponse
if len(paras) < 2:
    raise ValueError("Le document ne contient pas assez de texte (>=2 paragraphes).")

if len(paras) % 2 != 0:
    print("⚠️ Attention: nombre impair de paragraphes. Le dernier sera ignoré.")
    paras = paras[:-1]

rows = []
qid = 1
for i in range(0, len(paras), 2):
    q = paras[i]
    a = paras[i+1]
    rows.append({
        "id": f"Q{qid:03d}",
        "question": q,
        "expected_answer": a,
        "category": "",            # à remplir ensuite
        "must_refuse": "",         # TRUE/FALSE
        "must_cite_source": "",    # TRUE/FALSE
        "expected_sources": "",    # ex: page_001; guide_candidat_2025
        "scenario_id": "",         # ex: SCENARIO_1
        "notes": ""                # mots-clés, pièges, critères
    })
    qid += 1

df = pd.DataFrame(rows)
df.to_csv(OUT_CSV_PATH, index=False, encoding="utf-8-sig")

print("✅ CSV créé :", OUT_CSV_PATH)
print("Nb Q/R :", len(df))
df.head()

DOCX existe ? True
Nb paragraphes non vides: 131
⚠️ Attention: nombre impair de paragraphes. Le dernier sera ignoré.
✅ CSV créé : /content/drive/MyDrive/PIP_2025-2026_Groupe-1_Concours/Mohamed-Taha Belhaj - Analyse/questions_structured_template.csv
Nb Q/R : 65


,id,question,expected_answer,category,must_refuse,must_cite_source,expected_sources,scenario_id,notes
0,Q001,orientation selon profil,? : Est ce que le concours Experte ou expert e...,,,,,,
1,Q002,"rep : Pour le concours n°64, le grade mentionn...",info précise sur concours,,,,,,
2,Q003,? : Quelles sont les responsabilités de l'ingé...,rep : L'agent développera des outils numérique...,,,,,,
3,Q004,"données océanographiques hétérogènes, facilita...",processus biogéochimiques et biologiques marins.,,,,,,
4,Q005,"carrière, évolution, salaire",? : Quel salaire je peux prétendre en fin de c...,,,,,,


In [ ]:
from google.colab import files
files.download("/content/drive/MyDrive/PIP_2025-2026_Groupe-1_Concours/Mohamed-Taha Belhaj - Analyse/questions_structured_template.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>